# Transformer-Based LLM Chatbot — Kaggle Notebook

**Student:** Praveenkumar G.  
**Roll Number:** `24BAD090`  
**Course/Department:** B.Tech AI & Data Science  
**Experiment:** Build and test a Transformer-based chatbot using a pretrained LLM and Gradio.




## Part A — Environment setup and model selection




In [1]:
# Student details — included for assignment identification
STUDENT_NAME = "Praveenkumar G."
ROLL_NUMBER = "24BAD090"

print("Student:", STUDENT_NAME)
print("Roll Number:", ROLL_NUMBER)
print("Experiment: Transformer-Based LLM Chatbot")


Student: Praveenkumar G.
Roll Number: 24BAD090
Experiment: Transformer-Based LLM Chatbot


In [2]:
# Install only packages that are missing. This avoids unnecessary upgrades
# (and avoids contacting package servers when everything is already installed).
import importlib.util
import subprocess
import sys

PACKAGE_SPECS = {
    "transformers": "transformers>=4.45,<5",
    "accelerate": "accelerate",
    "gradio": "gradio>=4.44,<6",
}
missing_specs = [spec for module, spec in PACKAGE_SPECS.items()
                 if importlib.util.find_spec(module) is None]

if missing_specs:
    print("Missing packages:", ", ".join(missing_specs))
    print("Attempting to install missing packages. Internet access may be required.")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_specs])
    except Exception as exc:
        raise RuntimeError(
            "Package installation failed. If Kaggle Internet is denied, resolve the Kaggle "
            "account/Internet permission or use an environment where these packages are preinstalled."
        ) from exc
else:
    print("Required packages are already installed; skipping pip installation.")


Required packages are already installed; skipping pip installation.


In [3]:
import os
import sys
import json
import torch
import transformers
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU; generation may be slow.")


Python: 3.13.15
PyTorch: 2.11.0+cu128
Transformers: 5.16.1
CUDA available: True
GPU: Tesla T4


### Load the pretrained model and tokenizer

The loader first checks for a local Qwen2 model under `/kaggle/input`. If none is found, it downloads `Qwen/Qwen2.5-0.5B-Instruct` from Hugging Face. If the download is blocked and no local model exists, follow the error instructions shown by the cell.


In [4]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
# Optional: set this to your attached Kaggle Input model folder, e.g.
# LOCAL_MODEL_PATH = "/kaggle/input/qwen25-model"
# Leave empty to allow automatic detection.
LOCAL_MODEL_PATH = ""


def find_local_qwen_model():
    """Find a Qwen2 model folder attached under /kaggle/input, if present."""
    if LOCAL_MODEL_PATH:
        candidate = Path(LOCAL_MODEL_PATH)
        if (candidate / "config.json").is_file():
            return str(candidate)
        raise FileNotFoundError(
            f"LOCAL_MODEL_PATH was set to {LOCAL_MODEL_PATH!r}, but config.json was not found there."
        )

    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None

    for config_path in input_root.glob("**/config.json"):
        try:
            with config_path.open("r", encoding="utf-8") as file:
                config = json.load(file)
            if config.get("model_type") not in {"qwen2", "qwen2_moe"}:
                continue
            folder = config_path.parent
            has_tokenizer = any((folder / name).is_file() for name in
                                ("tokenizer.json", "tokenizer_config.json"))
            has_weights = any(
                any(folder.glob(pattern))
                for pattern in ("*.safetensors", "pytorch_model*.bin", "model.safetensors.index.json")
            )
            if has_tokenizer and has_weights:
                return str(folder)
        except (OSError, json.JSONDecodeError):
            continue
    return None


local_model_dir = find_local_qwen_model()
model_source = local_model_dir or MODEL_ID
using_local_files = local_model_dir is not None
print("Model source:", model_source)
print("Loading local files only:", using_local_files)

load_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
try:
    tokenizer = AutoTokenizer.from_pretrained(
        model_source,
        local_files_only=using_local_files
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_source,
        torch_dtype=load_dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
        local_files_only=using_local_files
    )
    if not torch.cuda.is_available():
        model.to("cpu")
    model.eval()
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Model and tokenizer loaded successfully.")
except Exception as exc:
    raise RuntimeError(
        "Could not load the model. If Internet is disabled, attach a Kaggle Dataset/Input "
        "containing the complete Qwen2.5-0.5B-Instruct files, set LOCAL_MODEL_PATH to that folder, "
        "and rerun this cell. Otherwise, resolve Kaggle Internet/account permissions and rerun. "
        f"Original error: {type(exc).__name__}: {exc}"
    ) from exc


Model source: Qwen/Qwen2.5-0.5B-Instruct
Loading local files only: False


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model and tokenizer loaded successfully.


## Part B — Prompt handling and response generation

The function validates empty prompts, formats messages with the model's chat template, keeps a limited amount of conversation history, and returns only the newly generated tokens.


In [5]:
SYSTEM_PROMPT = (
    "You are a helpful, polite AI assistant created as a student demonstration project. "
    "Give clear, concise answers. Use simple English when possible. If unsure, say so."
)


def generate_response(user_message, conversation_history=None, max_new_tokens=180):
    """Generate a reply from the loaded local Transformer model."""
    if not isinstance(user_message, str) or not user_message.strip():
        return "Please enter a non-empty message."

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Keep only the latest eight earlier messages so the prompt does not grow indefinitely.
    if isinstance(conversation_history, list):
        for item in conversation_history[-8:]:
            if not isinstance(item, dict):
                continue
            role = item.get("role")
            content = item.get("content")
            if role in ("user", "assistant") and isinstance(content, str) and content.strip():
                messages.append({"role": role, "content": content.strip()})

    messages.append({"role": "user", "content": user_message.strip()})

    try:
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=2048)
        input_device = model.get_input_embeddings().weight.device
        inputs = {key: value.to(input_device) for key, value in inputs.items()}

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=int(max_new_tokens),
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id
            )

        new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        return answer or "The model did not generate a response. Please try a different prompt."
    except Exception as exc:
        return f"Generation error ({type(exc).__name__}): {exc}"


# One simple test prompt
sample_prompt = "Hello! Introduce yourself in two sentences."
print("Student:", STUDENT_NAME, "| Roll No:", ROLL_NUMBER)
print("Prompt:", sample_prompt)
print("Response:", generate_response(sample_prompt))


Student: Praveenkumar G. | Roll No: 24BAD090
Prompt: Hello! Introduce yourself in two sentences.
Response: Hello! My name is [Your Name]. I'm an AI assistant designed to provide information and help with various tasks.


### Test the function with five different prompts

Run this cell and retain the output as evidence for Part B. The exact responses can vary between runs.


In [6]:
test_prompts = [
    "What is a large language model? Explain in simple English.",
    "Explain the difference between AI and machine learning in three points.",
    "Write a Python function to find the largest of three numbers.",
    "Give a short study plan for learning SQL in seven days.",
    "What are two limitations of AI chatbots?"
]

test_results = []
for number, prompt in enumerate(test_prompts, start=1):
    answer = generate_response(prompt, max_new_tokens=160)
    test_results.append({"prompt": prompt, "response": answer})
    print(f"TEST {number}")
    print("Prompt:", prompt)
    print("Response:", answer)
    print("-" * 80)

# Basic empty-input test
print("Empty input test:", generate_response("   "))


TEST 1
Prompt: What is a large language model? Explain in simple English.
Response: A large language model is a type of artificial intelligence (AI) that can generate human-like text based on input provided to it. It's like having a superpower that can write stories, explain things, or even solve problems with just one command.
Imagine you have a toy robot that can speak and understand commands. That's kind of what a large language model does - it has special algorithms and data stored in its brain to "understand" the commands given by humans and produce responses accordingly. This ability allows it to be very good at talking to people and understanding their thoughts and ideas.

In simpler terms: Imagine a computer program that can understand spoken instructions and create natural-sounding dialogue that sounds more human than a regular person would.
--------------------------------------------------------------------------------
TEST 2
Prompt: Explain the difference between AI and mac

## Part C — Build the chatbot interface with Gradio

The interface displays messages and uses Gradio's message history. A share URL requires network access; if sharing is unavailable, try the local launch option described below.


In [9]:

import gradio as gr

def gradio_chat(message, history):
    # Support both old and new Gradio history formats
    conversation_history = []

    for item in (history or []):
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content", "")
            if role in ("user", "assistant") and isinstance(content, str):
                conversation_history.append({
                    "role": role,
                    "content": content
                })
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_msg, assistant_msg = item
            if user_msg:
                conversation_history.append({
                    "role": "user",
                    "content": str(user_msg)
                })
            if assistant_msg:
                conversation_history.append({
                    "role": "assistant",
                    "content": str(assistant_msg)
                })

    return generate_response(
        message,
        conversation_history=conversation_history,
        max_new_tokens=220
    )

demo = gr.ChatInterface(
    fn=gradio_chat,
    title=f"LLM Chatbot — {STUDENT_NAME} ({ROLL_NUMBER})",
    description="Chat with a pretrained Qwen language model.",
    examples=[
        "Explain Python lists with a simple example.",
        "What is a database index?",
        "Give me three exam preparation tips."
    ]
)

print("Launching chatbot...")
demo.launch()


Launching chatbot...
* Running on local URL:  http://127.0.0.1:7861
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://d21240b6008400a618.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Part D — Testing, evaluation, and limitations




In [10]:
# Create requirements.txt for GitHub submission.
# Kaggle already provides PyTorch in most notebook images, but include it for a general setup.
requirements_text = """torch
transformers>=4.45,<5
accelerate
gradio>=4.44,<6
"""

with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write(requirements_text)

print("Created requirements.txt in the current working directory.")
print(requirements_text)


Created requirements.txt in the current working directory.
torch
transformers>=4.45,<5
accelerate
gradio>=4.44,<6

